# LinkedIn IT Job Scraper (Sri Lanka) - Keyword-by-Keyword Strategy

## Project: Skill-Aware Job Matching for Sri Lankan IT Professionals

This notebook scrapes **IT-specific** job postings from LinkedIn in Sri Lanka using a **keyword-by-keyword approach** for maximum coverage.

### New Scraping Strategy:
1. Individual Keyword Search: Each IT keyword is searched separately (100+ keywords)
2. Complete Pagination: Scrapes ALL available pages for each keyword
3. Progressive Saving: After each keyword, results are saved/appended to CSV
4. No Filter Combinations: Simplified approach - just keyword + location
5. Maximum Coverage: Gets every possible IT job by searching all relevant terms

### Search Keywords Include:
- Roles: developer, engineer, analyst, designer, manager, architect, etc.
- Specializations: frontend, backend, fullstack, devops, data scientist, QA, etc.
- Technologies: python, java, aws, azure, react, kubernetes, etc.
- Domains: software, IT, technology, security, cloud, data, AI, etc.

### Data Fields Extracted:
- Job ID, Title, Company Name, Location, Posted Date
- Job Description (full text)
- Experience Level, Employment Type, Job Function, Industries
- Required Skills (500+ IT skills extracted)
- Job Criteria, Number of Applicants, Job URL
- Search Keyword (for tracking which keyword found the job)
- Scraped Timestamp

### Why Keyword-by-Keyword?
- Maximum Coverage: Each keyword gets separate searches
- No Missed Jobs: Different keywords surface different jobs
- Progressive Saving: Data saved after each keyword (safe from failures)
- Resumable: Can stop and resume from last keyword
- Better Deduplication: Tracks which keywords found which jobs

### Output:
- Single CSV file that grows with each keyword
- Excel file (final dataset)
- Summary report with keyword statistics

In [31]:
# Step 1: Install Required Packages
!pip install requests beautifulsoup4 pandas tqdm openpyxl lxml

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Step 2: Import Libraries and Setup Configuration
import requests
from bs4 import BeautifulSoup
import math
import pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import logging
import time
import random
import re

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Multiple user agents to rotate
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
]

# Configuration - KEYWORD-BY-KEYWORD SCRAPING
CONFIG = {
    'location': 'Sri Lanka',
    'output_dir': 'data',
    'output_filename': 'linkedin_sri_lanka_IT_jobs_progressive.csv',
    'request_delay': (2, 4),  # Delay between requests (seconds)
    'page_delay': (3, 6),  # Delay between pages (seconds)
    'keyword_delay': (10, 15),  # Delay between keywords (seconds)
    'max_pages_per_keyword': 40,  # Max pages to scrape per keyword (LinkedIn typically shows ~1000 results max = 40 pages)
    
    # Comprehensive IT job search keywords - Each searched separately
    'search_keywords': [
        # Core Development Roles
        'software developer', 'software engineer', 'developer', 'engineer', 'programmer',
        'full stack developer', 'fullstack developer', 'backend developer', 'frontend developer',
        'web developer', 'mobile developer', 'application developer',
        
        # Specific Programming Languages
        'python developer', 'java developer', 'javascript developer', '.net developer',
        'php developer', 'ruby developer', 'golang developer', 'c# developer',
        'react developer', 'angular developer', 'vue developer', 'node.js developer',
        'ios developer', 'android developer', 'flutter developer', 'react native developer',
        
        # Architecture & Senior Roles
        'software architect', 'solutions architect', 'technical architect', 'cloud architect',
        'enterprise architect', 'system architect',
        'technical lead', 'tech lead', 'team lead', 'lead developer', 'lead engineer',
        'engineering manager', 'development manager', 'it manager',
        
        # DevOps & Infrastructure
        'devops engineer', 'devops', 'site reliability engineer', 'sre',
        'cloud engineer', 'infrastructure engineer', 'platform engineer', 'systems engineer',
        'kubernetes engineer', 'docker engineer', 'aws engineer', 'azure engineer', 'gcp engineer',
        'ci/cd engineer', 'automation engineer', 'build engineer', 'release engineer',
        
        # Data & Analytics
        'data scientist', 'data analyst', 'data engineer', 'business analyst',
        'business intelligence analyst', 'bi analyst', 'bi developer', 'analytics engineer',
        'data architect', 'big data engineer', 'etl developer', 'reporting analyst',
        
        # Machine Learning & AI
        'machine learning engineer', 'ml engineer', 'ai engineer', 'mlops engineer',
        'deep learning engineer', 'nlp engineer', 'computer vision engineer',
        'ai researcher', 'research scientist', 'data science',
        
        # Database & Administration
        'database administrator', 'dba', 'database developer', 'database engineer',
        'sql developer', 'mongodb developer', 'postgresql developer',
        'system administrator', 'systems administrator', 'sysadmin',
        'network administrator', 'it administrator', 'server administrator',
        
        # Quality Assurance & Testing
        'qa engineer', 'quality assurance engineer', 'test engineer', 'tester',
        'qa analyst', 'test analyst', 'sdet', 'automation engineer',
        'automation test engineer', 'manual tester', 'performance test engineer',
        'selenium tester', 'qa', 'quality assurance', 'testing',
        
        # Security
        'security engineer', 'cybersecurity engineer', 'information security analyst',
        'security analyst', 'security architect', 'security consultant',
        'penetration tester', 'ethical hacker', 'security', 'cybersecurity',
        
        # Support & Operations
        'technical support engineer', 'it support engineer', 'support engineer',
        'it support', 'technical support', 'help desk', 'service desk',
        'it specialist', 'it technician', 'desktop support',
        
        # UI/UX & Design
        'ui designer', 'ux designer', 'ui/ux designer', 'product designer',
        'ux researcher', 'interaction designer', 'visual designer',
        'web designer', 'graphic designer', 'ux', 'ui',
        
        # Product & Project Management
        'product manager', 'technical product manager', 'project manager',
        'program manager', 'scrum master', 'agile coach', 'product owner',
        'delivery manager', 'it project manager',
        
        # Specialized Technologies
        'salesforce developer', 'salesforce', 'sap consultant', 'sap',
        'oracle developer', 'tableau developer', 'power bi developer',
        'blockchain developer', 'smart contract developer', 'web3',
        'game developer', 'unity developer', 'embedded engineer',
        'iot engineer', 'rpa developer',
        
        # General IT Terms (will capture broad matches)
        'software', 'technology', 'IT', 'tech', 'digital',
        'analyst', 'consultant', 'specialist', 'coordinator',
    ]
}

def get_random_headers():
    """Get random headers to avoid detection"""
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.linkedin.com",
        "Connection": "keep-alive",
    }

logger.info("Configuration loaded successfully")
logger.info(f"Total search keywords: {len(CONFIG['search_keywords'])}")

Configuration loaded successfully
Mode: TEST (10 jobs)


In [33]:
# Step 3: Define HTTP Request Helper
def safe_get(url, max_retries=3):
    """Make HTTP GET request with retry logic"""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=get_random_headers(), timeout=10)
            
            if response.status_code == 429:
                logger.warning(f"Rate limited. Waiting... (Attempt {attempt + 1}/{max_retries})")
                time.sleep(30 * (attempt + 1))
                continue
            
            if response.status_code == 200:
                return response
            
            logger.warning(f"Status code {response.status_code} on attempt {attempt + 1}")
            
        except Exception as e:
            logger.error(f"Request error on attempt {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1))
    
    return None

# Test the function
test_url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?location=Sri%20Lanka&start=0"
response = safe_get(test_url)
if response:
    print(f"HTTP request function working! Status: {response.status_code}")
else:
    print("HTTP request function failed. Check your connection.")

HTTP request function working! Status: 200


In [ ]:
# Step 4: Define Skill Extraction Function
def extract_skills_from_text(text):
    """
    Extract IT skills from job description text
    """
    if not text:
        return []
    
    # Comprehensive IT skills dictionary
    SKILLS = [
        # Programming Languages
        'python', 'java', 'javascript', 'typescript', 'c++', 'c#', 'c', 'php', 'ruby', 'go', 'golang', 
        'rust', 'kotlin', 'swift', 'scala', 'r', 'perl', 'matlab', 'julia', 'dart', 'objective-c',
        'vb.net', 'visual basic', 'cobol', 'fortran', 'haskell', 'elixir', 'clojure',
        
        # Web Technologies - Frontend
        'html', 'html5', 'css', 'css3', 'sass', 'scss', 'less', 'tailwind', 'tailwind css', 
        'bootstrap', 'material ui', 'mui', 'chakra ui', 'ant design',
        'react', 'react.js', 'reactjs', 'angular', 'angular.js', 'vue', 'vue.js', 'vuejs',
        'next.js', 'nextjs', 'nuxt', 'nuxt.js', 'svelte', 'ember', 'backbone',
        'jquery', 'webpack', 'vite', 'parcel', 'rollup', 'gulp', 'grunt',
        
        # Web Technologies - Backend
        'node.js', 'nodejs', 'express', 'express.js', 'fastify', 'nest.js', 'nestjs',
        'django', 'flask', 'fastapi', 'pyramid', 'tornado',
        'spring', 'spring boot', 'spring mvc', 'hibernate', 'struts',
        'asp.net', '.net', '.net core', 'asp.net mvc', 'blazor',
        'laravel', 'symfony', 'codeigniter', 'cakephp', 'yii',
        'ruby on rails', 'rails', 'sinatra',
        'phoenix', 'gin', 'echo', 'fiber',
        
        # Databases - SQL
        'sql', 'mysql', 'postgresql', 'postgres', 'oracle', 'oracle db', 'sql server', 'mssql',
        'mariadb', 'sqlite', 'db2', 'sybase', 'snowflake', 'teradata', 'amazon redshift', 'redshift',
        
        # Databases - NoSQL
        'mongodb', 'cassandra', 'couchdb', 'redis', 'memcached', 'dynamodb', 'neo4j', 
        'elasticsearch', 'elastic search', 'firebase', 'firestore', 'cosmosdb', 'hbase',
        'riak', 'couchbase', 'ravendb',
        
        # Cloud Platforms & Services
        'aws', 'amazon web services', 'azure', 'microsoft azure', 'gcp', 'google cloud', 
        'google cloud platform', 'ibm cloud', 'oracle cloud', 'alibaba cloud', 'digital ocean',
        'heroku', 'netlify', 'vercel', 'cloudflare',
        'ec2', 's3', 'lambda', 'rds', 'dynamodb', 'cloudfront', 'route53', 'ecs', 'eks',
        'azure functions', 'azure devops', 'app service', 'cosmos db',
        'cloud functions', 'cloud run', 'cloud storage', 'bigquery', 'dataflow', 'pub/sub',
        
        # DevOps & Infrastructure
        'docker', 'kubernetes', 'k8s', 'openshift', 'helm', 'istio', 'service mesh',
        'jenkins', 'gitlab ci', 'github actions', 'circle ci', 'travis ci', 'bamboo', 'teamcity',
        'ci/cd', 'continuous integration', 'continuous deployment',
        'terraform', 'terragrunt', 'ansible', 'puppet', 'chef', 'saltstack', 'cloudformation',
        'vagrant', 'packer', 'consul', 'vault', 'nomad',
        'prometheus', 'grafana', 'elk', 'elk stack', 'logstash', 'kibana', 'splunk', 'datadog',
        'new relic', 'dynatrace', 'nagios', 'zabbix', 'fluentd',
        
        # Version Control & Collaboration
        'git', 'github', 'gitlab', 'bitbucket', 'svn', 'subversion', 'mercurial', 'perforce',
        'git flow', 'trunk based development',
        
        # Data Science & Machine Learning
        'machine learning', 'ml', 'deep learning', 'artificial intelligence', 'ai',
        'data science', 'data analysis', 'data analytics', 'statistical analysis',
        'tensorflow', 'pytorch', 'keras', 'scikit-learn', 'sklearn', 'xgboost', 'lightgbm', 'catboost',
        'pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'plotly', 'bokeh',
        'jupyter', 'jupyter notebook', 'jupyterlab', 'google colab',
        'nlp', 'natural language processing', 'computer vision', 'cv', 'opencv',
        'reinforcement learning', 'neural networks', 'cnn', 'rnn', 'lstm', 'gru', 'transformer',
        'bert', 'gpt', 'hugging face', 'spacy', 'nltk', 'gensim',
        
        # MLOps & AI Engineering
        'mlops', 'ml ops', 'mlflow', 'kubeflow', 'airflow', 'apache airflow', 'prefect', 'dagster',
        'sagemaker', 'vertex ai', 'azure ml', 'databricks', 'dvc', 'data version control',
        'model deployment', 'model monitoring', 'feature store', 'feast',
        'bentoml', 'seldon', 'kserve', 'triton',
        
        # Data Engineering
        'data engineering', 'etl', 'elt', 'data pipeline', 'data warehouse', 'data lake',
        'big data', 'apache spark', 'spark', 'pyspark', 'hadoop', 'mapreduce', 'hdfs',
        'hive', 'pig', 'presto', 'trino', 'apache flink', 'flink', 'kafka', 'apache kafka',
        'stream processing', 'batch processing', 'apache beam', 'dataflow',
        'dbt', 'data build tool', 'airbyte', 'fivetran', 'stitch', 'talend', 'informatica',
        'apache nifi', 'luigi', 'oozie',
        
        # Business Intelligence & Analytics
        'business intelligence', 'bi', 'data visualization', 'tableau', 'power bi', 'powerbi',
        'looker', 'qlik', 'qlikview', 'qlik sense', 'metabase', 'superset', 'apache superset',
        'sisense', 'domo', 'thoughtspot', 'microstrategy', 'sap business objects',
        'google data studio', 'data studio', 'redash', 'mode analytics',
        
        # UI/UX Design
        'ui', 'ux', 'ui/ux', 'user interface', 'user experience', 'ux design', 'ui design',
        'figma', 'sketch', 'adobe xd', 'invision', 'axure', 'balsamiq', 'framer',
        'prototyping', 'wireframing', 'user research', 'usability testing', 'interaction design',
        'visual design', 'design systems', 'design thinking', 'responsive design', 'mobile design',
        'adobe photoshop', 'photoshop', 'adobe illustrator', 'illustrator', 'zeplin', 'miro',
        
        # Mobile Development
        'android', 'android development', 'ios', 'ios development', 'mobile development',
        'react native', 'flutter', 'xamarin', 'ionic', 'cordova', 'phonegap',
        'swift', 'swiftui', 'objective-c', 'kotlin', 'java android',
        'android studio', 'xcode', 'app store', 'play store', 'firebase',
        
        # Game Development
        'game development', 'unity', 'unity3d', 'unreal engine', 'unreal', 'godot',
        'game design', '2d', '3d', 'blender', 'maya', '3ds max', 'substance painter',
        
        # Blockchain & Web3
        'blockchain', 'web3', 'ethereum', 'solidity', 'smart contracts', 'defi', 'nft',
        'cryptocurrency', 'bitcoin', 'hyperledger', 'truffle', 'hardhat', 'metamask',
        
        # Security & Cybersecurity
        'cybersecurity', 'security', 'information security', 'infosec', 'application security',
        'network security', 'penetration testing', 'ethical hacking', 'vulnerability assessment',
        'owasp', 'siem', 'ids', 'ips', 'firewall', 'waf', 'ssl', 'tls', 'encryption',
        'oauth', 'saml', 'jwt', 'authentication', 'authorization', 'identity management',
        'pci dss', 'gdpr', 'hipaa', 'iso 27001', 'soc 2',
        
        # Testing & QA
        'testing', 'qa', 'quality assurance', 'test automation', 'automated testing',
        'unit testing', 'integration testing', 'end-to-end testing', 'e2e testing',
        'selenium', 'cypress', 'playwright', 'puppeteer', 'webdriver', 'appium',
        'jest', 'mocha', 'jasmine', 'pytest', 'junit', 'testng', 'nunit',
        'postman', 'rest assured', 'jmeter', 'gatling', 'locust', 'k6',
        'performance testing', 'load testing', 'stress testing', 'tdd', 'bdd', 'cucumber',
        
        # API & Integration
        'api', 'rest', 'restful', 'rest api', 'restful api', 'graphql', 'grpc', 'soap',
        'microservices', 'microservice architecture', 'api gateway', 'api management',
        'webhook', 'websocket', 'sse', 'server-sent events',
        
        # Project Management & Methodologies
        'agile', 'scrum', 'kanban', 'lean', 'waterfall', 'devops', 'safe', 'xp',
        'jira', 'confluence', 'trello', 'asana', 'monday.com', 'azure boards',
        'project management', 'product management', 'roadmap', 'sprint planning',
        
        # Operating Systems & Infrastructure
        'linux', 'unix', 'ubuntu', 'centos', 'rhel', 'red hat', 'debian', 'fedora',
        'windows', 'windows server', 'macos', 'bash', 'shell scripting', 'powershell',
        'virtualization', 'vmware', 'virtualbox', 'hyper-v', 'kvm', 'proxmox',
        
        # Networking
        'networking', 'tcp/ip', 'dns', 'dhcp', 'vpn', 'load balancing', 'cdn',
        'nginx', 'apache', 'iis', 'haproxy', 'envoy', 'traefik',
        
        # Content Management
        'cms', 'wordpress', 'drupal', 'joomla', 'contentful', 'strapi', 'sanity',
        'headless cms', 'sitecore', 'aem', 'adobe experience manager',
        
        # E-commerce
        'e-commerce', 'ecommerce', 'shopify', 'magento', 'woocommerce', 'prestashop',
        'bigcommerce', 'salesforce commerce cloud',
        
        # CRM & Marketing
        'salesforce', 'crm', 'hubspot', 'marketo', 'pardot', 'dynamics 365',
        'seo', 'sem', 'google analytics', 'google ads', 'facebook ads', 'marketing automation',
        
        # ERP & Enterprise
        'erp', 'sap', 'oracle erp', 'netsuite', 'odoo', 'microsoft dynamics',
        
        # Messaging & Real-time
        'rabbitmq', 'activemq', 'zeromq', 'nats', 'pulsar', 'mqtt', 'amqp',
        'websockets', 'socket.io', 'signalr',
        
        # Search & Indexing
        'solr', 'apache solr', 'algolia', 'typesense', 'meilisearch',
        
        # Containers & Orchestration
        'containerization', 'container orchestration', 'docker swarm', 'rancher', 'portainer',
        
        # Soft Skills (Technical Context)
        'problem solving', 'debugging', 'troubleshooting', 'code review', 'pair programming',
        'technical documentation', 'system design', 'architecture design', 'scalability',
        'performance optimization', 'refactoring', 'clean code', 'design patterns',
        'solid principles', 'microservices patterns', 'distributed systems',
        
        # Emerging Technologies
        'iot', 'internet of things', 'edge computing', 'quantum computing',
        'augmented reality', 'ar', 'virtual reality', 'vr', 'mixed reality', 'mr',
        'serverless', 'faas', 'low code', 'no code', 'rpa', 'robotic process automation',
    ]
    
    text_lower = text.lower()
    found_skills = []
    
    for skill in SKILLS:
        pattern = r'\b' + re.escape(skill.lower()) + r'\b'
        if re.search(pattern, text_lower):
            found_skills.append(skill.title())
    
    return list(set(found_skills))

Test extraction: Found 13 skills
Skills: Airflow, Bi, Figma, Kubernetes, Mlops, Power Bi, Python, Spark, Tableau, Ui, Ui/Ux, Ux, Ux Design


In [ ]:
# Step 5: Define IT Relevance Check and Job Scraping Functions

def is_it_related_job(title, company, description, industries):
    """
    Check if a job is IT-related based on title, description, and industries
    Returns True if job is relevant to IT industry
    """
    if not title:
        return False
    
    # Comprehensive IT-related keywords in job titles
    it_title_keywords = [
        # Development & Programming
        'software', 'developer', 'dev', 'engineer', 'programmer', 'coder', 'coding',
        'architect', 'technical lead', 'tech lead', 'lead developer', 'senior developer',
        'junior developer', 'software engineer', 'software developer', 'application developer',
        
        # Specialized Development
        'frontend', 'front-end', 'front end', 'backend', 'back-end', 'back end',
        'fullstack', 'full-stack', 'full stack', 'web developer', 'web engineer',
        'mobile developer', 'ios developer', 'android developer', 'app developer',
        'game developer', 'embedded developer', 'firmware engineer',
        
        # DevOps & Infrastructure
        'devops', 'dev ops', 'sre', 'site reliability', 'infrastructure engineer',
        'cloud engineer', 'platform engineer', 'systems engineer', 'build engineer',
        'release engineer', 'deployment engineer', 'automation engineer',
        
        # Cloud & Platform
        'cloud', 'aws', 'azure', 'gcp', 'cloud architect', 'cloud consultant',
        'cloud specialist', 'solutions architect', 'infrastructure architect',
        
        # Data & Analytics
        'data', 'analyst', 'data analyst', 'data scientist', 'scientist',
        'data engineer', 'ml engineer', 'machine learning', 'ml', 'ai',
        'artificial intelligence', 'deep learning', 'analytics', 'business analyst',
        'data architect', 'big data', 'etl developer', 'bi developer',
        'business intelligence', 'bi', 'reporting analyst', 'insights analyst',
        
        # Database & Administration
        'database', 'dba', 'database administrator', 'sql', 'database engineer',
        'database developer', 'data warehouse', 'administrator', 'admin',
        'system administrator', 'sys admin', 'sysadmin', 'it administrator',
        
        # Quality Assurance & Testing
        'qa', 'qc', 'quality assurance', 'quality control', 'tester', 'test engineer',
        'testing', 'automation tester', 'sdet', 'qa engineer', 'qa analyst',
        'test analyst', 'quality engineer', 'test automation',
        
        # Security
        'security', 'cybersecurity', 'infosec', 'information security',
        'security engineer', 'security analyst', 'security architect',
        'penetration tester', 'ethical hacker', 'security consultant',
        'compliance', 'risk analyst', 'security operations',
        
        # System & Network
        'system', 'systems', 'network', 'network engineer', 'network administrator',
        'it support', 'technical support', 'support engineer', 'help desk',
        'service desk', 'it specialist', 'it technician', 'technical',
        'infrastructure', 'operations', 'it operations',
        
        # Management & Leadership
        'product manager', 'project manager', 'program manager', 'technical manager',
        'engineering manager', 'it manager', 'delivery manager', 'scrum master',
        'agile coach', 'product owner', 'team lead', 'technical lead',
        
        # Design & UX
        'ui', 'ux', 'ui/ux', 'designer', 'design', 'user experience',
        'user interface', 'product designer', 'ux designer', 'ui designer',
        'graphic designer', 'web designer', 'interaction designer',
        'visual designer', 'ux researcher', 'design lead',
        
        # Specialized Roles
        'consultant', 'specialist', 'coordinator', 'technical writer',
        'solutions engineer', 'pre-sales', 'technical consultant',
        'implementation engineer', 'integration engineer', 'api developer',
        'microservices', 'blockchain developer', 'integration specialist',
        
        # Data Science & ML Specific
        'nlp', 'computer vision', 'research scientist', 'ai researcher',
        'mlops', 'machine learning engineer', 'data science', 'statistician',
        'quantitative analyst', 'algorithm engineer', 'modeling',
        
        # Emerging Tech
        'blockchain', 'web3', 'iot', 'robotics', 'automation',
        'rpa', 'low code', 'no code', 'chatbot', 'conversational ai',
        
        # General IT Terms
        'technology', 'tech', 'digital', 'informatics', 'computing',
        'information technology', 'it professional', 'technologist',
        'software development', 'application development', 'development',
        
        # Tools & Frameworks (Role-specific)
        'salesforce', 'sap', 'oracle', 'microsoft', 'servicenow',
        'workday', 'tableau', 'power bi', 'sharepoint',
        
        # Methodologies
        'agile', 'scrum', 'kanban', 'lean', 'ci/cd', 'continuous integration',
    ]
    
    # Comprehensive IT-related industries
    it_industries_keywords = [
        # Core IT
        'software', 'technology', 'it services', 'information technology',
        'computer software', 'computer', 'computing', 'tech',
        
        # Internet & Digital
        'internet', 'web', 'online', 'digital', 'e-commerce', 'ecommerce',
        'e-learning', 'edtech', 'education technology',
        
        # Telecommunications
        'telecommunications', 'telecom', 'wireless', 'networking',
        'communication', 'mobile', 'broadband',
        
        # Financial Tech
        'fintech', 'financial technology', 'financial services',
        'banking technology', 'payment', 'blockchain',
        
        # Cloud & SaaS
        'saas', 'paas', 'iaas', 'cloud computing', 'cloud services',
        'cloud', 'platform', 'hosting', 'data center',
        
        # Specialized Tech
        'artificial intelligence', 'machine learning', 'ai', 'ml',
        'data analytics', 'big data', 'analytics', 'cybersecurity',
        'information security', 'gaming', 'video games', 'entertainment software',
        
        # Business Tech
        'enterprise software', 'business intelligence', 'crm', 'erp',
        'hr technology', 'marketing technology', 'martech',
        'automation', 'workflow', 'productivity software',
        
        # Emerging Tech
        'iot', 'internet of things', 'robotics', 'autonomous',
        'augmented reality', 'virtual reality', 'ar', 'vr', 'metaverse',
        'cryptocurrency', 'web3', 'blockchain technology',
        
        # Media & Content Tech
        'media technology', 'digital media', 'content management',
        'streaming', 'social media', 'advertising technology', 'adtech',
        
        # Health & Science Tech
        'healthtech', 'health technology', 'medical software', 'biotech software',
        'research software', 'scientific software', 'pharmaceutical technology',
        
        # Others
        'consulting', 'outsourcing', 'managed services', 'integration',
        'systems integration', 'startup', 'innovation', 'research and development',
    ]
    
    # Check title
    title_lower = title.lower()
    if any(keyword in title_lower for keyword in it_title_keywords):
        return True
    
    # Check industries
    if industries:
        industries_lower = industries.lower()
        if any(keyword in industries_lower for keyword in it_industries_keywords):
            return True
    
    # Check description (if available)
    if description:
        desc_lower = description.lower()
        # Look for IT-related terms in description
        it_desc_keywords = ['software', 'programming', 'coding', 'development', 'technical', 'technology']
        match_count = sum(1 for keyword in it_desc_keywords if keyword in desc_lower)
        if match_count >= 2:  # At least 2 IT keywords in description
            return True
    
    return False

def get_job_details(job_id):
    """Get detailed job information from LinkedIn API"""
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
    
    try:
        response = safe_get(url)
        if not response:
            return {}
        
        soup = BeautifulSoup(response.text, 'html.parser')
        details = {}
        
        # Extract description
        desc_elem = soup.find("div", {"class": "show-more-less-html__markup"})
        if desc_elem:
            description = desc_elem.get_text(separator=" ", strip=True)
            details["description"] = description
            # Extract skills from description
            skills = extract_skills_from_text(description)
            details["required_skills"] = ', '.join(skills) if skills else None
        else:
            details["description"] = None
            details["required_skills"] = None
        
        # Extract job criteria (stored as single field and individual fields)
        criteria_list = soup.find("ul", {"class": "description__job-criteria-list"})
        criteria_text_list = []
        
        if criteria_list:
            for item in criteria_list.find_all("li"):
                header = item.find("h3", {"class": "description__job-criteria-subheader"})
                value = item.find("span", {"class": "description__job-criteria-text"})
                
                if header and value:
                    header_text = header.text.strip()
                    value_text = value.text.strip()
                    criteria_text_list.append(f"{header_text}: {value_text}")
                    
                    if "Seniority level" in header_text:
                        details["experience_level"] = value_text
                    elif "Employment type" in header_text:
                        details["employment_type"] = value_text
                    elif "Job function" in header_text:
                        details["job_function"] = value_text
                    elif "Industries" in header_text:
                        details["industries"] = value_text
        
        # Store combined job criteria
        details["job_criteria"] = " | ".join(criteria_text_list) if criteria_text_list else None
        
        # Extract number of applicants
        try:
            num_applicants_elem = soup.find("span", {"class": "num-applicants__caption"})
            if not num_applicants_elem:
                num_applicants_elem = soup.find("figcaption", {"class": "num-applicants__caption"})
            
            if num_applicants_elem:
                details["num_applicants"] = num_applicants_elem.get_text(strip=True)
            else:
                details["num_applicants"] = None
        except:
            details["num_applicants"] = None
        
        # Ensure all required fields exist with None if not found
        for field in ["experience_level", "employment_type", "job_function", "industries"]:
            if field not in details:
                details[field] = None
        
        return details
        
    except Exception as e:
        logger.error(f"Error fetching details for job {job_id}: {e}")
        return {}

def extract_job_card_data(card, search_keyword=""):
    """Extract data from a job card HTML element"""
    try:
        # Get job link and ID
        job_link = card.find("a", {"class": "base-card__full-link"})
        if not job_link:
            return None
        
        job_url = job_link.get('href', '').split('?')[0]
        job_id = job_url.split('-')[-1] if job_url else None
        
        if not job_id:
            return None
        
        # Extract basic info
        title_elem = card.find("h3", {"class": "base-search-card__title"})
        company_elem = card.find("h4", {"class": "base-search-card__subtitle"})
        location_elem = card.find("span", {"class": "job-search-card__location"})
        time_elem = card.find("time")
        
        job_data = {
            "job_id": job_id,
            "title": title_elem.text.strip() if title_elem else None,
            "company": company_elem.text.strip() if company_elem else None,
            "location": location_elem.text.strip() if location_elem else None,
            "posted_date": time_elem.get("datetime") if time_elem else None,
            "job_url": job_url,
            "search_keyword": search_keyword,
            "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        # Get detailed info (with delay to avoid rate limiting)
        time.sleep(random.uniform(1, 2))
        details = get_job_details(job_id)
        job_data.update(details)
        
        return job_data
        
    except Exception as e:
        logger.error(f"Error extracting job card data: {e}")
        return None

print("Scraping functions defined successfully")

Scraping functions defined successfully


In [ ]:
# Step 6: Main Scraping Function for Single Keyword
def scrape_jobs_by_keyword(keyword, seen_ids=None):
    """
    Scrape LinkedIn IT jobs for a single keyword in Sri Lanka
    Goes through ALL available pages for the keyword
    """
    if seen_ids is None:
        seen_ids = set()
    
    jobs_data = []
    location = CONFIG['location'].replace(' ', '%20')
    keyword_encoded = keyword.replace(' ', '%20')
    
    # Simple URL - just keyword and location (no filters)
    base_url = (
        f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?"
        f"keywords={keyword_encoded}&"
        f"location={location}&"
        f"start={{}}"
    )
    
    max_pages = CONFIG['max_pages_per_keyword']
    
    logger.info(f"Scraping keyword: '{keyword}'")
    
    for page in range(max_pages):
        try:
            url = base_url.format(page * 25)
            response = safe_get(url)
            
            if not response:
                logger.warning(f"  Failed to get page {page} for '{keyword}'")
                break
            
            soup = BeautifulSoup(response.text, 'html.parser')
            job_cards = soup.find_all("div", {"class": "base-card"})
            
            if not job_cards:
                logger.info(f"  No more jobs found at page {page} for '{keyword}'")
                break
            
            page_new_jobs = 0
            for card in job_cards:
                job_data = extract_job_card_data(card, search_keyword=keyword)
                
                if job_data and job_data['job_id'] not in seen_ids:
                    # Check if IT-related
                    if is_it_related_job(
                        job_data.get('title'),
                        job_data.get('company'),
                        job_data.get('description'),
                        job_data.get('industries')
                    ):
                        seen_ids.add(job_data['job_id'])
                        jobs_data.append(job_data)
                        page_new_jobs += 1
            
            if page_new_jobs > 0:
                logger.info(f"  Page {page}: Found {page_new_jobs} new IT jobs")
            
            # Delay between pages
            time.sleep(random.uniform(*CONFIG['page_delay']))
            
        except Exception as e:
            logger.error(f"  Error on page {page} for '{keyword}': {e}")
            continue
    
    logger.info(f"Keyword '{keyword}' complete: {len(jobs_data)} new jobs")
    return jobs_data

print("Keyword-based scraping function ready")

Main scraping function ready


## Execute Keyword-by-Keyword Scraping

**Run this cell to scrape IT jobs using each keyword separately.**

This will:
- Search each keyword one by one (100+ keywords)
- Scrape ALL available pages for each keyword
- Save/append results to CSV after each keyword
- Deduplicate across all keywords by job_id
- Accept nulls for missing data fields
- Resume-friendly: Can stop and restart without losing data

**Estimated time:** 2-4 hours (depending on results per keyword)

**Note**: Progress is saved after each keyword, so you can safely stop and resume.

In [ ]:
# Step 7: Execute Keyword-by-Keyword Scraping with Progressive Saving
import os

print("="*60)
print("KEYWORD-BY-KEYWORD SCRAPING - SRI LANKAN IT JOBS")
print("="*60)
print(f"Total keywords: {len(CONFIG['search_keywords'])}")
print(f"Max pages per keyword: {CONFIG['max_pages_per_keyword']}")
print("="*60 + "\n")

# Setup output
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(exist_ok=True)
csv_filepath = output_dir / CONFIG['output_filename']

# Track progress
seen_ids = set()
total_jobs_collected = 0
keyword_stats = []

# Load existing data if CSV exists (for resuming)
if csv_filepath.exists():
    print(f"Found existing CSV: {csv_filepath}")
    existing_df = pd.read_csv(csv_filepath)
    seen_ids = set(existing_df['job_id'].astype(str).tolist())
    total_jobs_collected = len(existing_df)
    print(f"Loaded {total_jobs_collected} existing jobs\n")
else:
    print(f"Starting fresh: {csv_filepath}\n")

# Iterate through each keyword
for idx, keyword in enumerate(CONFIG['search_keywords'], 1):
    print(f"\n[{idx}/{len(CONFIG['search_keywords'])}] Searching: '{keyword}'")
    
    # Scrape jobs for this keyword
    new_jobs = scrape_jobs_by_keyword(keyword, seen_ids)
    
    if new_jobs:
        # Convert to DataFrame
        df_new = pd.DataFrame(new_jobs)
        
        # Append to CSV (create if doesn't exist)
        if not csv_filepath.exists():
            df_new.to_csv(csv_filepath, index=False, encoding='utf-8-sig', mode='w')
            print(f"Created CSV with {len(df_new)} jobs")
        else:
            df_new.to_csv(csv_filepath, index=False, encoding='utf-8-sig', mode='a', header=False)
            print(f"Appended {len(df_new)} jobs to CSV")
        
        total_jobs_collected += len(df_new)
        keyword_stats.append({
            'keyword': keyword,
            'jobs_found': len(df_new),
            'total_so_far': total_jobs_collected
        })
    else:
        keyword_stats.append({
            'keyword': keyword,
            'jobs_found': 0,
            'total_so_far': total_jobs_collected
        })
    
    print(f"Running total: {total_jobs_collected} unique jobs")
    
    # Delay between keywords
    if idx < len(CONFIG['search_keywords']):
        delay = random.uniform(*CONFIG['keyword_delay'])
        print(f"Waiting {delay:.1f}s before next keyword...")
        time.sleep(delay)

print("\n" + "="*60)
print("SCRAPING COMPLETE - PERFORMING FINAL DEDUPLICATION")
print("="*60)

# Load final dataset and perform thorough deduplication
if csv_filepath.exists():
    df_full = pd.read_csv(csv_filepath)
    initial_count = len(df_full)
    
    print(f"\nInitial records: {initial_count}")
    
    # Deduplicate by job_id (keep first occurrence)
    df_full = df_full.drop_duplicates(subset=['job_id'], keep='first')
    final_count = len(df_full)
    duplicates_removed = initial_count - final_count
    
    print(f"After deduplication: {final_count}")
    print(f"Duplicates removed: {duplicates_removed}")
    
    # Save the final deduplicated dataset
    final_csv_filepath = output_dir / "linkedin_sri_lanka_IT_jobs_final.csv"
    df_full.to_csv(final_csv_filepath, index=False, encoding='utf-8-sig')
    print(f"\nFinal CSV saved: {final_csv_filepath}")
    
    # Remove the progressive CSV file (keep only final)
    if csv_filepath != final_csv_filepath and csv_filepath.exists():
        try:
            os.remove(csv_filepath)
            print(f"Removed intermediate file: {csv_filepath}")
        except Exception as e:
            print(f"Could not remove intermediate file: {e}")
    
    print(f"\nSuccessfully scraped {final_count} unique IT jobs")
    print(f"Data shape: {df_full.shape}")
    
    # Show top keywords
    print(f"\nTop 10 Keywords by Jobs Found:")
    df_stats = pd.DataFrame(keyword_stats)
    top_keywords = df_stats[df_stats['jobs_found'] > 0].nlargest(10, 'jobs_found')
    for _, row in top_keywords.iterrows():
        print(f"  {row['keyword']}: {row['jobs_found']} jobs")
    
else:
    print("\nNo jobs scraped. Check errors above.")
    df_full = pd.DataFrame()
    final_csv_filepath = None

FULL SCRAPE NOT ENABLED
Set PROCEED_WITH_FULL_SCRAPE = True to start full scraping


## Generate Final Files

Final deduplication and file generation:

After scraping completes, the system will:
1. Load all scraped data from progressive CSV
2. Remove duplicates by job_id (keeping first occurrence)
3. Save single final CSV: linkedin_sri_lanka_IT_jobs_final.csv
4. Delete intermediate progressive file
5. Generate Excel file with timestamp
6. Create comprehensive summary report

Result: Only 3 files in the data/ folder:
- 1 CSV file (deduplicated)
- 1 Excel file
- 1 Summary report

In [ ]:
# Step 8: Generate Final Excel and Summary Report
if not df_full.empty and final_csv_filepath:
    # Create output directory
    output_dir = Path(CONFIG['output_dir'])
    output_dir.mkdir(exist_ok=True)
    
    # Generate timestamp for final files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_filename = output_dir / f"linkedin_sri_lanka_IT_jobs_{timestamp}.xlsx"
    
    # Save to Excel
    try:
        df_full.to_excel(excel_filename, index=False, engine='openpyxl')
        print(f"Excel file created: {excel_filename}")
    except Exception as e:
        print(f"Excel save failed: {e}")
    
    # Save comprehensive summary statistics
    summary_filename = output_dir / f"scraping_summary_{timestamp}.txt"
    with open(summary_filename, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("LINKEDIN IT JOB SCRAPING SUMMARY\n")
        f.write("Keyword-by-Keyword Strategy\n")
        f.write("="*60 + "\n\n")
        f.write(f"Scrape Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total IT Jobs Scraped: {len(df_full)}\n")
        f.write(f"Total Keywords Searched: {len(CONFIG['search_keywords'])}\n")
        f.write(f"Location: {CONFIG['location']}\n")
        f.write(f"Strategy: Individual keyword search with full pagination\n\n")
        f.write("="*60 + "\n")
        f.write("TOP 20 KEYWORDS BY JOBS FOUND\n")
        f.write("="*60 + "\n\n")
        
        if 'search_keyword' in df_full.columns:
            keyword_counts = df_full['search_keyword'].value_counts().head(20)
            for kw, count in keyword_counts.items():
                f.write(f"{kw}: {count} jobs\n")
        
        f.write("\n" + "="*60 + "\n")
        f.write("DATA COMPLETENESS\n")
        f.write("="*60 + "\n\n")
        
        for col in df_full.columns:
            missing = df_full[col].isna().sum()
            coverage = ((len(df_full) - missing) / len(df_full)) * 100
            f.write(f"{col}: {coverage:.1f}% ({len(df_full) - missing}/{len(df_full)})\n")
        
        f.write("\n" + "="*60 + "\n")
        f.write("TOP 10 COMPANIES\n")
        f.write("="*60 + "\n\n")
        if 'company' in df_full.columns:
            top_companies = df_full['company'].value_counts().head(10)
            for company, count in top_companies.items():
                f.write(f"{company}: {count} jobs\n")
        
        f.write("\n" + "="*60 + "\n")
        f.write("TOP 10 JOB TITLES\n")
        f.write("="*60 + "\n\n")
        if 'title' in df_full.columns:
            top_titles = df_full['title'].value_counts().head(10)
            for title, count in top_titles.items():
                f.write(f"{title}: {count} postings\n")
    
    print(f"Summary report: {summary_filename}")
    
    print("\n" + "="*60)
    print("FINAL STATISTICS")
    print("="*60)
    print(f"Total Jobs: {len(df_full)}")
    print(f"Unique Companies: {df_full['company'].nunique() if 'company' in df_full.columns else 'N/A'}")
    print(f"Unique Job Titles: {df_full['title'].nunique() if 'title' in df_full.columns else 'N/A'}")
    print(f"Keywords Used: {len(CONFIG['search_keywords'])}")
    print(f"Keywords with Results: {df_full['search_keyword'].nunique() if 'search_keyword' in df_full.columns else 'N/A'}")
    if 'posted_date' in df_full.columns and df_full['posted_date'].notna().any():
        print(f"Date Range: {df_full['posted_date'].min()} to {df_full['posted_date'].max()}")
    print(f"Data completeness: {df_full.notna().sum().sum() / (len(df_full) * len(df_full.columns)) * 100:.1f}%")
    print("="*60)
    
    print("\nFILES GENERATED")
    print("="*60)
    print(f"1. CSV: {final_csv_filepath}")
    print(f"2. Excel: {excel_filename}")
    print(f"3. Summary: {summary_filename}")
    print("="*60)
    
else:
    print("No data to process. Please run scraping first.")

Data saved to CSV: data\linkedin_sri_lanka_IT_jobs_20251014_125602.csv
   Total records: 10
Data saved to Excel: data\linkedin_sri_lanka_IT_jobs_20251014_125602.xlsx
Summary saved to: data\scraping_summary_20251014_125602.txt

FINAL STATISTICS
Total Jobs: 10
Unique Companies: 9
Unique Job Titles: 8
Date Range: 2025-06-17 to 2025-10-13


In [ ]:
# Step 9: Exploratory Data Analysis
if not df_full.empty:
    print("="*60)
    print("EXPLORATORY DATA ANALYSIS")
    print("="*60)
    
    print("\n[Top 15 Keywords by Jobs Found]")
    if 'search_keyword' in df_full.columns:
        print(df_full['search_keyword'].value_counts().head(15))
    
    print("\n[Top 10 Companies by Job Postings]")
    if 'company' in df_full.columns:
        print(df_full['company'].value_counts().head(10))
    
    print("\n[Top 10 Job Titles]")
    if 'title' in df_full.columns:
        print(df_full['title'].value_counts().head(10))
    
    print("\n[Location Distribution]")
    if 'location' in df_full.columns:
        print(df_full['location'].value_counts().head(10))
    
    print("\n[Experience Level Distribution]")
    if 'experience_level' in df_full.columns:
        exp_counts = df_full['experience_level'].value_counts()
        if not exp_counts.empty:
            print(exp_counts)
    
    print("\n[Employment Type Distribution]")
    if 'employment_type' in df_full.columns:
        emp_counts = df_full['employment_type'].value_counts()
        if not emp_counts.empty:
            print(emp_counts)
    
    print("\n[Most In-Demand Skills]")
    if 'required_skills' in df_full.columns:
        all_skills = []
        for skills_str in df_full['required_skills'].dropna():
            if skills_str:
                skills_list = [s.strip() for s in skills_str.split(',')]
                all_skills.extend(skills_list)
        
        if all_skills:
            from collections import Counter
            skill_counts = Counter(all_skills)
            print("\nTop 20 Most In-Demand IT Skills:")
            for skill, count in skill_counts.most_common(20):
                percentage = (count / len(df_full)) * 100
                print(f"  {skill}: {count} jobs ({percentage:.1f}%)")
    
    print("\n[Data Completeness by Column]")
    completeness = ((df_full.notna().sum() / len(df_full)) * 100).sort_values(ascending=False)
    for col, pct in completeness.items():
        print(f"  {col}: {pct:.1f}%")
    
    print("\n[Keyword Effectiveness]")
    if 'search_keyword' in df_full.columns:
        print(f"Total keywords searched: {len(CONFIG['search_keywords'])}")
        print(f"Keywords with results: {df_full['search_keyword'].nunique()}")
        print(f"Effectiveness rate: {(df_full['search_keyword'].nunique() / len(CONFIG['search_keywords']) * 100):.1f}%")
    
    print("\n" + "="*60)
    print("Data exploration complete")
    print("="*60)
else:
    print("No data available for exploration")

EXPLORATORY DATA ANALYSIS

[Top 10 Companies by Job Postings]
company
SenzMate AIoT Lab                     2
Ontash                                1
qlub                                  1
OrangeHRM                             1
Nimi                                  1
InEight                               1
Fintechnology Asia Pacific (FINAP)    1
DoMedia                               1
Cemex Software Solutions              1
Name: count, dtype: int64

[Top 10 Job Titles]
title
Software Engineer / Associate Software Engineer    2
Software Engineer                                  2
Frontend Developer                                 1
SOFTWARE DEVELOPER                                 1
Software Engineer - Backend Python Developer       1
Software Engineer (.NET)                           1
Front End Developer                                1
Full Stack Web Developer                           1
Name: count, dtype: int64

[Location Distribution]
location
Colombo, Western Province, Sri La

## Summary & Next Steps

### Successfully Completed - Keyword-by-Keyword Strategy:
- Individual Keyword Search: Each of 100+ keywords searched separately
- Complete Pagination: All available pages scraped per keyword
- Progressive Saving: Data saved after each keyword (safe from failures)
- Maximum Coverage: Different keywords surface different jobs
- No Filters: Simple search (keyword + location only)
- Smart Deduplication: job_id tracking across all keywords
- Resumable: Can stop and restart without losing progress
- Comprehensive IT Focus: 100+ role types, technologies, specializations
- 500+ Skill Extraction: Modern IT skills from job descriptions
- Null-Friendly: Missing data gracefully handled

### Dataset Features:

**Search Keywords Used (100+ total)**:
- Roles: developer, engineer, analyst, designer, manager, architect
- Specializations: frontend, backend, fullstack, devops, data scientist
- Technologies: python, java, react, kubernetes, aws, azure
- Domains: security, QA, UI/UX, database, cloud, AI/ML

**Data Fields Collected**:
- Core: job_id, title, company, location, posted_date, job_url
- Details: description, experience_level, employment_type, job_function, industries
- Skills: required_skills (500+ IT skills extracted)
- Metadata: search_keyword, job_criteria, num_applicants, scraped_at

### Why This Approach is Better:

| Aspect | Old Approach | New Keyword-by-Keyword |
|--------|--------------|------------------------|
| Coverage | Combined OR keywords | Each keyword searched separately |
| Pagination | Limited by filters | ALL pages per keyword |
| Resilience | Lose all if fails | Progressive save per keyword |
| Resumable | No | Yes - can restart anytime |
| Deduplication | Per filter combo | Across all keywords |
| Flexibility | Fixed filters | Pure keyword-based |
| Max Jobs | ~4000 | 100+ keywords x 1000 jobs = 100,000+ potential |

### Output Files:
1. Final CSV: linkedin_sri_lanka_IT_jobs_final.csv (deduplicated, single file)
2. Excel: linkedin_sri_lanka_IT_jobs_{timestamp}.xlsx
3. Summary Report: scraping_summary_{timestamp}.txt (keyword stats, top findings)

Note: Only ONE final CSV file is saved after complete deduplication. Any intermediate files are automatically removed.

### For Your ML Project:

#### 1. You Now Have Maximum IT Job Coverage
- Every possible IT keyword searched
- All pages scraped for each
- Comprehensive dataset for training

#### 2. Keyword Analysis Possible
- Which keywords yield most jobs?
- Which roles are most in-demand?
- Emerging vs traditional roles

#### 3. Ready for ML Pipeline
```python
# Example preprocessing
- Skill matrix (500+ binary features)
- Job title embeddings (BERT)
- Company clustering
- Keyword-based job categorization
- Time series analysis (by search_keyword)
```

#### 4. Resume-Based Job Matching
```python
# Your ML model can:
1. Extract skills from candidate resume
2. Match against scraped jobs
3. Rank by skill overlap
4. Recommend learning paths for skill gaps
5. Show which keywords/roles match best
```

### Important Notes:
1. Scraping Time: 2-4 hours for all 100+ keywords (with delays)
2. Progressive Save: Safe to stop anytime - resume later
3. Rate Limits: Built-in delays respect LinkedIn's servers
4. Deduplication: Automatic across all keywords
5. Educational Use: For academic projects only

### SDG 8 Alignment:
- Comprehensive Job Market View: All IT roles covered
- Better Job Matching: Keyword-level granularity
- Skill Development: Identifies in-demand skills by role type
- Career Planning: Shows demand across different specializations

Your dataset is now ready for advanced ML analysis!